**Vitals BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Vitals"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# SCHEMA
# =========================================

schema = [

    bigquery.SchemaField(
        "Vitals_Key",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Booking_Key",
        "STRING"
    ),

    bigquery.SchemaField(
        "Patient_U_ID",
        "STRING"
    ),

    bigquery.SchemaField(
        "Visit_Date",
        "DATE"
    ),

    bigquery.SchemaField(
        "Systolic_BP",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Diastolic_BP",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Heart_Rate",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Pulse",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Resp_Rate",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Temp",
        "NUMERIC"
    ),

    bigquery.SchemaField(
        "O2_Sat",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "RBS",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Wt",
        "NUMERIC"
    )

]

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.clustering_fields = [
    "Patient_U_ID",
    "Booking_Key"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Vitals


**Vitals Postgres**

In [2]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS vitals_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Vitals (

    Vitals_Key TEXT PRIMARY KEY
    DEFAULT (
        'VK' ||
        LPAD(
            nextval('vitals_no_seq')::TEXT,
            4,
            '0'
        )
    ),

    Booking_Key TEXT,

    Patient_U_ID TEXT,

    Visit_Date DATE,

    Systolic_BP INTEGER,

    Diastolic_BP INTEGER,

    Heart_Rate INTEGER,

    Pulse INTEGER,

    Resp_Rate INTEGER,

    Temp NUMERIC(5,1),

    O2_Sat INTEGER,

    RBS INTEGER,

    Wt NUMERIC(6,1)

);
"""

cursor.execute(create_table_query)

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_vitals_patient
ON Vitals(Patient_U_ID);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_vitals_booking
ON Vitals(Booking_Key);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Vitals")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Vitals


**Sync BigQuery to  Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

query = """

SELECT

    Vitals_Key,
    Booking_Key,
    Patient_U_ID,
    Visit_Date,
    Systolic_BP,
    Diastolic_BP,
    Heart_Rate,
    Pulse,
    Resp_Rate,
    Temp,
    O2_Sat,
    RBS,
    Wt

FROM `depihealthnux.Depihealthnux_Main.Vitals`

ORDER BY Vitals_Key

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Vitals;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Vitals (

            Vitals_Key,
            Booking_Key,
            Patient_U_ID,
            Visit_Date,
            Systolic_BP,
            Diastolic_BP,
            Heart_Rate,
            Pulse,
            Resp_Rate,
            Temp,
            O2_Sat,
            RBS,
            Wt

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'vitals_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Vitals_Key,'VK','')
                    AS INTEGER
                )
            )
            FROM Vitals
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Vitals
ORDER BY Vitals_Key
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [Vitals_Key, Booking_Key, Patient_U_ID, Visit_Date, Systolic_BP, Diastolic_BP, Heart_Rate, Pulse, Resp_Rate, Temp, O2_Sat, RBS, Wt]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgres To BigQuery**

In [4]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    Vitals_Key,
    Booking_Key,
    Patient_U_ID,
    Visit_Date,
    Systolic_BP,
    Diastolic_BP,
    Heart_Rate,
    Pulse,
    Resp_Rate,
    Temp,
    O2_Sat,
    RBS,
    Wt

FROM Vitals

ORDER BY Vitals_Key

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Vitals",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Vitals`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_13296\3728648147.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Empty DataFrame
Columns: [vitals_key, booking_key, patient_u_id, visit_date, systolic_bp, diastolic_bp, heart_rate, pulse, resp_rate, temp, o2_sat, rbs, wt]
Index: []
Rows Retrieved: 0
Uploaded 0 rows

First 3 Rows From BigQuery
Empty DataFrame
Columns: [vitals_key, booking_key, patient_u_id, visit_date, systolic_bp, diastolic_bp, heart_rate, pulse, resp_rate, temp, o2_sat, rbs, wt]
Index: []
